# Clase 030 — Strings vectorizados

**Parte 0** · VanderPlas cap. 3 § 3.11.

> 🎯 Limpiar texto sin `apply(lambda)`. El accessor `.str` es vectorizado y NaN-aware.

> ⏱️ ~60 min

## ⚙️ Setup

In [ ]:
import pandas as pd
import numpy as np

## 1️⃣ El accessor `.str`

Pandas expone métodos string vectorizados via `.str`:

```python
s.str.lower()
s.str.strip()
s.str.replace('a', 'b')
s.str.contains('patron', regex=True)
s.str.extract(r'(\d+)')
s.str.split(',', expand=True)
s.str.len()
```

**Ventaja sobre `apply(lambda)`**: vectorizado (5–10× más rápido) y maneja NaN automáticamente.

In [ ]:
emails = pd.Series([' Ana@Example.com', 'BOB@gmail.com  ', np.nan, 'cris@FOO.io', 'dan@example.com'])
print('original:')
print(emails)

limpio = emails.str.lower().str.strip()
print('\nnormalizado:')
print(limpio)
print('\nNaN se preserva sin error.')

## 2️⃣ Regex con `.str.extract`

Pattern con grupos `()` → DataFrame con una columna por grupo:

In [ ]:
dominios = limpio.str.extract(r'@(?P<dominio>[\w\.]+)$')
print(dominios)

# Combinar con el original
print('\ncombinado:')
print(pd.concat([limpio.rename('email'), dominios], axis=1))

## 3️⃣ `.str.contains` + regex para filtros

In [ ]:
df = pd.DataFrame({
    'email': ['ana@example.com', 'bob@gmail.com', 'cris@empresa.es', 'dan@yahoo.com'],
    'desc' : ['Comentario URGENTE', 'normal', 'urgente revisar', 'bug critico']
})

# Email corporativo (no es de proveedor mainstream)
mainstream = r'gmail|yahoo|hotmail|outlook'
df['corp'] = ~df['email'].str.contains(mainstream, case=False, regex=True)

# Descripción urgente (case-insensitive)
df['urgente'] = df['desc'].str.contains('urgente', case=False)

print(df)

## 4️⃣ `.str.split(expand=True)` — desnormalizar

In [ ]:
nombres = pd.Series(['Ana García', 'Bob Smith Jr', 'Cris López-Mora'])
partes = nombres.str.split(' ', n=1, expand=True)
partes.columns = ['nombre', 'apellido']
print(partes)

## 5️⃣ Dtypes string vs object

```python
pd.Series(['a','b','c'], dtype='string')  # nullable, NA-aware
pd.Series(['a','b','c'])                  # default: object (mezcla Python)
```

El dtype `'string'` es el moderno: integra con `pd.NA`, optimizaciones futuras.

## 6️⃣ `Categorical` — para baja cardinalidad

Si una columna tiene pocos valores únicos comparado al total de filas (ej: país, sexo, tipo), `Categorical` ahorra memoria masivamente y acelera `groupby`/`sort`:

In [ ]:
rng = np.random.default_rng(42)
N = 100_000
paises = rng.choice(['ES', 'CL', 'MX', 'AR', 'CO'], N)

s_obj = pd.Series(paises)
s_cat = pd.Series(paises, dtype='category')

print(f'object   : {s_obj.memory_usage(deep=True)/1024:.0f} KB')
print(f'category : {s_cat.memory_usage(deep=True)/1024:.0f} KB')
print(f'ratio    : {s_obj.memory_usage(deep=True)/s_cat.memory_usage(deep=True):.1f}×')

## ✅ Checklist

- [ ] Uso `.str.lower()`, `.str.strip()`, etc. — no `apply(lambda)`
- [ ] Sé que `.str` maneja NaN automáticamente
- [ ] Aplico regex con `.str.extract` y `.str.contains`
- [ ] Uso `.str.split(expand=True)` para desnormalizar
- [ ] Uso `Categorical` para columnas de baja cardinalidad

## 📝 Homework

Ver `README.md`. CSV contactos: normalizar email, extraer dominio, separar nombre, flag corp, Categorical país.

## 📖 Definiciones y características

**Accessor `.str`**

Espacio de nombres en Series con métodos string vectorizados. Análogos a los de Python (`.lower()`, `.split()`, `.replace()`) pero aplicados elementwise y NaN-aware (propagan NaN sin error).

**`.str.extract(pattern)`**

Aplica regex con grupos `()` y devuelve DataFrame con una columna por grupo. Soporta grupos nombrados (`(?P<dominio>...)`).

**`.str.split(sep, expand=True)`**

Divide cada string y opcionalmente expande a DataFrame de columnas. Útil para denormalizar 'Apellido, Nombre' → 2 cols.

**dtype `'string'` (nullable)**

Versión moderna del dtype para texto. Diferencias con `object`: NA-aware (usa `pd.NA`), futuras optimizaciones. Recomendado en pandas 2+.

**`Categorical`**

Dtype para columnas con cardinalidad baja (pocos valores únicos). Almacena cada valor como entero + diccionario. **Ahorra ~10× memoria** y acelera groupby/sort.

## ⚠️ Errores comunes

| Síntoma / mensaje | Causa y cómo arreglar |
|---|---|
| `'NoneType' has no attribute 'lower'` al hacer `s.apply(str.lower)` | Hay NaN/None en la Series. **Fix**: usa `.str.lower()` (accessor) — maneja NaN automáticamente. |
| Regex no captura nada con `.str.extract` | Falta `()` para definir grupo, o el pattern no matchea. **Fix**: testa el regex en https://regex101.com con un sample primero. |
| `.str.contains('foo')` lanza error con NaN | Por default, `na=NaN` propaga. **Fix**: `s.str.contains('foo', na=False)` trata NaN como False. |
| Convertí a `Categorical` y el sort sale alfabético | Categorical por default es no-ordenado. **Fix**: `pd.Categorical(s, categories=['bajo','medio','alto'], ordered=True)` para imponer orden. |
| `.str.split(',')` da listas, no columnas | Sin `expand=True`. **Fix**: `s.str.split(',', expand=True)` devuelve DataFrame con una columna por parte. |

## ❓ Preguntas frecuentes

**❓ ¿`.str.lower()` o `.apply(str.lower)`?**

**`.str.lower()`** siempre — vectorizado, maneja NaN, mucho más rápido en N grande. `apply` es loop Python disfrazado.

**❓ ¿Cuándo convertir a `Categorical`?**

Cuando la cardinalidad es baja (~<5% de N filas) y vas a hacer groupby/sort. Para 100k filas de 5 países: enorme ganancia. Para 100k filas de 80k strings únicos: no ayuda.

**❓ ¿`'string'` o `object` dtype?**

**`'string'`** para código nuevo (NA-aware). **`object`** sigue siendo default por compat. Conviértelo explícito: `df['col'] = df['col'].astype('string')`.

**❓ ¿Regex case-insensitive?**

`s.str.contains('foo', case=False)` o `flags=re.IGNORECASE`. También `s.str.lower().str.contains('foo')` (más explícito).

**❓ ¿Cómo elimino acentos?**

Pandas no trae nativo. Usa `unidecode` o `s.str.normalize('NFKD').str.encode('ascii', 'ignore').str.decode('ascii')`.

## 🔗 Referencias

- VanderPlas cap. 3 § 3.11
- [pandas Text](https://pandas.pydata.org/docs/user_guide/text.html)
- [pandas Categorical](https://pandas.pydata.org/docs/user_guide/categorical.html)

➡️ **Siguiente:** [031 — Series de tiempo](../031-pandas-series-de-tiempo-resampling-rolling/README.md)

## ✅ Soluciones de los ejercicios

A continuación, cada ejercicio de la sección `🧪 Ejercicios` del README resuelto y comentado. Todo el código es **ejecutable sin conexión** (datos sintéticos) e incluye `assert`/`print` para que compruebes el resultado. Intenta resolverlos por tu cuenta antes de mirar la solución.

**Ej. 1 — `lower` + `strip`** para normalizar emails.

In [ ]:
import pandas as pd
emails = pd.Series(['  Ana@MAIL.com ', 'BETO@Foo.COM', 'caro@bar.org  '])
norm = emails.str.lower().str.strip()
print(norm.tolist())
assert norm.iloc[0] == 'ana@mail.com' 

**Ej. 2 — Extraer el dominio** con regex.

In [ ]:
dominios = norm.str.extract(r'@(.+)$')[0]
print(dominios.tolist())
assert dominios.iloc[0] == 'mail.com' 

**Ej. 3 — Split de nombre completo** en dos columnas.

In [ ]:
nombres = pd.Series(['Ana Garcia', 'Beto Perez', 'Caro Ruiz'])
partes = nombres.str.split(' ', n=1, expand=True)
partes.columns = ['nombre', 'apellido']
print(partes)
assert partes.loc[0, 'apellido'] == 'Garcia' 

**Ej. 4 — Filtro `contains`** (case-insensitive).

In [ ]:
desc = pd.Series(['pedido urgente', 'envio normal', 'URGENTE revisar', 'ok'])
mask = desc.str.contains('urgente', case=False)
print(desc[mask].tolist())
assert mask.sum() == 2

**Ej. 5 — `Categorical`** y ahorro de memoria.

In [ ]:
import numpy as np
rng = np.random.default_rng(30)
s = pd.Series(rng.choice(['norte', 'sur', 'este', 'oeste', 'centro'], size=100_000))
mem_obj = s.memory_usage(deep=True)
mem_cat = s.astype('category').memory_usage(deep=True)
print(f'object: {mem_obj/1024:.1f} KB | category: {mem_cat/1024:.1f} KB')
assert mem_cat < mem_obj